# Study 872 — Nominal-Price Illusion 🪙

**A $10 stock and a $500 stock — is the cheap one an over-priced lottery ticket?**

Kumar (2009) and Birru & Wang (2016) argue the raw **nominal share price** is a pure
*money illusion*: value = price × shares, so the dollar price of one share carries **no**
information about a firm — yet retail lottery demand piles into low-priced names, and
investors over-estimate how much a cheap share can "grow". If that demand over-prices
cheap-looking stocks, low-priced names should carry the lottery look (more volatility,
more right-skew) and **lower risk-adjusted returns**. We sort a liquid US cross-section on
its nominal price level (2010-01-04 → 2026-06-30, 50 names).

*Numbers below are the frozen headline (`docs/results.md`, fingerprint
`357fd262912f`); the live cells run the fast synthetic control. Survivorship + proxy:
current-membership mega-caps are rarely cheap, and the Close is split-back-adjusted — an
honest nominal-price proxy. Honest low power.*


## 1. The idea in one picture

Two firms worth the same can trade at $10 or $500 a share — the number is set by an arbitrary share count, nothing real. But a $10 share *feels* cheaper and *feels* like it has more room to run, so lottery-hunting retail money crowds into low-priced names. If they over-pay, the cheap names should be **over-priced lotteries**: more volatile, more right-skewed, and — the payoff prediction — **lower risk-adjusted returns**. Sort on the price level; short the cheap, buy the dear.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=2.92, t_nw=3.01, lo_bps=8.75, hi_bps=5.82, lo_sharpe=1.15, hi_sharpe=0.82,
         cheap_name='T', cheap_val=20, dear_name='CAT', dear_val=1063)
print('price range at as-of: cheapest %s $%d .. priciest %s $%d (no single-digit names)'
      % (R['cheap_name'], R['cheap_val'], R['dear_name'], R['dear_val']))
print('long cheap / short dear spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  cheap book %+.2f bps vs dear book %+.2f bps' % (R['lo_bps'], R['hi_bps']))
print('  cheap Sharpe %+.2f vs dear Sharpe %+.2f' % (R['lo_sharpe'], R['hi_sharpe']))

price range at as-of: cheapest T $20 .. priciest CAT $1063 (no single-digit names)
long cheap / short dear spread: +2.92 bps/day (NW t = +3.01)
  cheap book +8.75 bps vs dear book +5.82 bps
  cheap Sharpe +1.15 vs dear Sharpe +0.82


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world (`edge>0`: cheap names look lottery-like *and* under-earn) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`: cheap names still look lottery-like but the price predicts nothing about returns). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from nominal_price import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=872, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.0016, seed=872, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should be NEGATIVE: cheap under-earns)' % planted['t_nw'])
print('planted lottery look: cheap vol %.1f%% skew %+.2f  vs  dear vol %.1f%% skew %+.2f'
      % (planted['lo_vol']*100, planted['lo_skew'], planted['hi_vol']*100, planted['hi_skew']))

null world   : spread NW t = -0.65  (should be ~0)
planted world: spread NW t = -3.69  (should be NEGATIVE: cheap under-earns)
planted lottery look: cheap vol 19.8% skew +0.89  vs  dear vol 6.3% skew +0.33


## 3. The honest verdict — the illusion does *not* pay here

On this liquid mega-cap tape the long-cheap / short-dear spread is **+2.92 bps/day** with NW *t* = **+3.01** — significant, but with the **opposite sign** to the claim: here the low-priced names actually *out-earned* the expensive ones, and with a **higher** Sharpe (+1.15 vs +0.82). The catch is baked into the universe — mega-caps are **rarely cheap** (the cheapest name is ~$20, no true low-dollar lottery stocks exist here), so the retail-lottery segment the theory targets is simply absent, and within mega-caps the lower-dollar names are the more value-tilted ones that quietly did well. The seeded synthetic control recovers a *planted* under-earn relation cleanly, so this is a genuine sign-reversal, not a bug. **Signal: None** (the claimed edge is absent), **Tradability: Mirage** (even the sign-flip book dies at cost).